# Fine-tuning de BETO en Colab (GPU gratis) â€” TFM

Entrena la clasificaciÃ³n de subtemas con **GPU gratuita** usando el mismo dataset de la app.
Flujo: exportar dataset desde la app â†’ entrenar aquÃ­ â†’ descargar los pesos â†’ importarlos al servicio ML.

**Runtime â†’ Cambiar tipo de entorno de ejecuciÃ³n â†’ GPU** antes de empezar.

In [ ]:
# Conserva NumPy/scikit-learn de Colab; degradarlos rompe paquetes preinstalados.
!pip -q install --upgrade transformers==4.57.6 datasets==4.4.1 accelerate==1.12.0
import numpy, sklearn, transformers, datasets, accelerate, torch
print('numpy', numpy.__version__, '| sklearn', sklearn.__version__)
print('transformers', transformers.__version__, '| datasets', datasets.__version__, '| accelerate', accelerate.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE')

## 1. Cargar el dataset
OpciÃ³n A: descargar desde la API (`GET /api/datasets/{id}/export` con tu JWT).
OpciÃ³n B: subir el JSON exportado a mano.

In [ ]:
import json, requests

USE_API = False  # True = descargar de la API; False = subir archivo
API_URL = 'https://tfm-api.onrender.com'  # tu backend
TOKEN = 'PEGA_TU_JWT'                      # de /api/auth/login
DATASET_ID = '9e99a778-7fab-4e29-bd8b-45466c3728dc'

if USE_API:
    r = requests.get(f'{API_URL}/api/datasets/{DATASET_ID}/export',
                     headers={'Authorization': f'Bearer {TOKEN}'})
    r.raise_for_status()
    data = r.json()
else:
    from google.colab import files
    up = files.upload()  # sube el JSON del endpoint /export
    data = json.loads(next(iter(up.values())).decode('utf-8'))

labels = sorted(data['labels'])
items = data['items']
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print(len(items), 'ejemplos Â·', len(labels), 'clases:', labels)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

BASE = 'dccuchile/bert-base-spanish-wwm-cased'
MAX_LEN = 192
tokenizer = AutoTokenizer.from_pretrained(BASE)

def split(name):
    rows = [r for r in items if r['split'] == name] or items
    return Dataset.from_dict({'text': [r['text'] for r in rows],
                              'label': [label2id[r['label']] for r in rows]})

def tok(b):
    return tokenizer(b['text'], truncation=True, max_length=MAX_LEN)

ds_train = split('train').map(tok, batched=True)
ds_val = split('val').map(tok, batched=True)
ds_test = split('test').map(tok, batched=True)

## 2. Entrenar (class weights + early stopping)

In [ ]:
import numpy as np, torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=len(labels), id2label=id2label, label2id=label2id)

counts = np.bincount([label2id[r['label']] for r in items if r['split']=='train'], minlength=len(labels))
w = torch.tensor([(counts.sum()/(len(labels)*c)) if c>0 else 0. for c in counts], dtype=torch.float)

class WT(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **k):
        y = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, y, weight=w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metrics(p):
    y = p.label_ids; pred = np.argmax(p.predictions, axis=-1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    return {'accuracy': accuracy_score(y, pred), 'f1_macro': f1}

args = TrainingArguments(output_dir='out', num_train_epochs=4, per_device_train_batch_size=16,
    per_device_eval_batch_size=16, learning_rate=2e-5, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1_macro', seed=42, report_to=[])
trainer = WT(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
             compute_metrics=metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

## 3. MÃ©tricas en test

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)

# Mismo bloque de mÃ©tricas que ml/trainer.py â€” deja `final_metrics` listo para importar a la app.
eval_split = 'test' if any(r['split'] == 'test' for r in items) else 'val'
pred = trainer.predict(ds_test)
y_true, y_pred = pred.label_ids, np.argmax(pred.predictions, axis=-1)

acc = accuracy_score(y_true, y_pred)
pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
pr_c, rc_c, f1_c, sup_c = precision_recall_fscore_support(
    y_true, y_pred, labels=range(len(labels)), zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))

final_metrics = {
    'accuracy': float(acc), 'precision_macro': float(pr),
    'recall_macro': float(rc), 'f1_macro': float(f1),
    'per_class': [{'label': labels[i], 'precision': float(pr_c[i]), 'recall': float(rc_c[i]),
                   'f1': float(f1_c[i]), 'support': int(sup_c[i])} for i in range(len(labels))],
    'confusion_matrix': cm.tolist(), 'labels': labels, 'split': eval_split,
}
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
print('Matriz de confusiÃ³n:\n', cm)

## 4. Guardar y descargar los pesos

1. Descarga `modelo_vN.zip` y descomprÃ­melo en `apps/ml/storage/models/vN/` del servicio ML.
2. Registra la versiÃ³n en la app con la celda 5 (`POST /api/models/import`) â€” asÃ­ aparece en
   la vista **Versiones** con sus mÃ©tricas.
3. ActÃ­vala desde la vista Versiones (o `POST /api/models/{id}/activate`).

In [ ]:
import os
OUT = 'beto-v1-gold-source-aware'
os.makedirs(OUT, exist_ok=True)
trainer.save_model(OUT); tokenizer.save_pretrained(OUT)
with open(f'{OUT}/labels.json','w',encoding='utf-8') as f:
    json.dump({'id2label': {str(i): l for i,l in id2label.items()}, 'max_len': MAX_LEN}, f, ensure_ascii=False, indent=2)
with open(f'{OUT}/metrics.json','w',encoding='utf-8') as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2)
with open(f'{OUT}/experiment.json','w',encoding='utf-8') as f:
    json.dump({'versionTag': OUT, 'datasetId': data['dataset']['id'], 'baseModel': BASE,
               'maxLen': MAX_LEN, 'trainingArguments': trainer.args.to_dict()},
              f, ensure_ascii=False, indent=2, default=str)
!zip -qr {OUT}.zip $OUT
from google.colab import files; files.download(f'{OUT}.zip')

## 5. Registrar la versiÃ³n en la app (`POST /api/models/import`)

Con los pesos ya descomprimidos en `apps/ml/storage/models/vN/`, esta celda imprime el JSON
para registrar la versiÃ³n (o la registra directamente si descomentas el `requests.post`).

In [ ]:
VERSION_TAG = 'beto-v1-gold-source-aware'  # ajusta (v2, v3, â€¦) â€” debe coincidir con la carpeta del paso 4

import_payload = {
    'versionTag': VERSION_TAG,
    'artifactPath': f'storage/models/{VERSION_TAG}',
    'hyperparams': {'epochs': 4, 'lr': 2e-5, 'batch_size': 16, 'max_len': MAX_LEN, 'seed': 42},
    'metrics': final_metrics,
}
import_payload['datasetId'] = data['dataset']['id']

print(json.dumps(import_payload, ensure_ascii=False, indent=2))

# Descomenta para registrarla directamente (requiere TOKEN vigente de /api/auth/login):
# r = requests.post(f'{API_URL}/api/models/import', json=import_payload,
#                   headers={'Authorization': f'Bearer {TOKEN}'})
# r.raise_for_status(); print('VersiÃ³n registrada:', r.json()['id'])

In [ ]:
# Ejecutar antes de registrar: evidencia academica y comparacion reproducible con TF-IDF.
import hashlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

test_rows = [r for r in items if r['split'] == 'test']
train_rows = [r for r in items if r['split'] == 'train']
if not test_rows or not train_rows:
    raise ValueError('El snapshot definitivo debe contener train y test')
label_ids = list(range(len(labels)))

def metric_report(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=label_ids, average='macro', zero_division=0)
    _, _, f1w, _ = precision_recall_fscore_support(y, p, labels=label_ids, average='weighted', zero_division=0)
    pc, rc_c, fc, support = precision_recall_fscore_support(y, p, labels=label_ids, zero_division=0)
    return {'accuracy': float(accuracy_score(y, p)), 'precision_macro': float(pr),
            'recall_macro': float(rc), 'f1_macro': float(f1), 'f1_weighted': float(f1w),
            'per_class': [{'label': labels[i], 'precision': float(pc[i]), 'recall': float(rc_c[i]),
                           'f1': float(fc[i]), 'support': int(support[i])} for i in label_ids],
            'confusion_matrix': confusion_matrix(y, p, labels=label_ids).tolist(),
            'labels': labels, 'split': 'test'}

def bootstrap_ci(y, p, iterations=2000, seed=42):
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); scores = []
    for _ in range(iterations):
        idx = rng.integers(0, len(y), size=len(y))
        scores.append(precision_recall_fscore_support(y[idx], p[idx], labels=label_ids,
                      average='macro', zero_division=0)[2])
    low, high = np.percentile(scores, [2.5, 97.5])
    return {'method': 'bootstrap_percentile_95', 'iterations': iterations, 'seed': seed,
            'low': float(low), 'high': float(high)}

vectorizer = TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1, 2),
                             min_df=2, max_features=50000, sublinear_tf=True)
x_train = vectorizer.fit_transform([r['text'] for r in train_rows])
x_test = vectorizer.transform([r['text'] for r in test_rows])
tfidf = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42, solver='lbfgs')
tfidf.fit(x_train, [r['label'] for r in train_rows])
tfidf_pred = np.asarray([label2id[label] for label in tfidf.predict(x_test)])
tfidf_metrics = metric_report(y_true, tfidf_pred)

# update conserva la referencia ya usada por import_payload.
final_metrics.update(metric_report(y_true, y_pred))
final_metrics['f1_macro_ci95'] = bootstrap_ci(y_true, y_pred)
final_metrics['results_by_source_type'] = {}
for source_type in sorted({r.get('sourceType') for r in test_rows if r.get('sourceType')}):
    idx = [i for i, row in enumerate(test_rows) if row.get('sourceType') == source_type]
    final_metrics['results_by_source_type'][source_type] = {
        'n_samples': len(idx), 'beto': metric_report(y_true[idx], y_pred[idx])}
canonical = json.dumps(data, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
final_metrics['baseline'] = {'name': 'tfidf_logistic_regression',
    'f1_macro': tfidf_metrics['f1_macro'],
    'dataset_sha256': hashlib.sha256(canonical.encode('utf-8')).hexdigest()}
print('BETO F1 macro:', final_metrics['f1_macro'])
print('TF-IDF F1 macro:', tfidf_metrics['f1_macro'])
print('Supera baseline:', final_metrics['f1_macro'] > tfidf_metrics['f1_macro'])
print('IC95 BETO:', final_metrics['f1_macro_ci95'])
import_payload['metrics'] = final_metrics
print(json.dumps(import_payload, ensure_ascii=False, indent=2))
# Registra solo despues de ejecutar esta celda y copiar los pesos al servicio ML:
# r = requests.post(f'{API_URL}/api/models/import', json=import_payload,
#                   headers={'Authorization': f'Bearer {TOKEN}'})
# r.raise_for_status(); print('Version registrada:', r.json()['id'])
